# Complainify AI — 06 : Sentiment Analysis (Lexicon-Based)

Sentiment lexicons, negation + intensifier handling, unknown-word handling, and the sentiment-to-priority pipeline.


In [1]:
import re


---
## PART 6: SENTIMENT ANALYSIS — LEXICON-BASED

Unlike the classifier which uses probability math, sentiment analysis uses:
1. **Word lists** — manually scored positive/negative words
2. **Negation flipping** — `"not good"` → negative
3. **Intensifier amplification** — `"very bad"` → more negative

**IMPORTANT:** For sentiment analysis, we do NOT remove stopwords!
The word `"not"` is critical for negation detection.

In [2]:
# ============================================================
# SENTIMENT LEXICONS
# ============================================================

NEGATION_WORDS = {'not', 'no', 'never', 'neither', 'nor', 'none', 'nothing',
                  'nobody', 'nowhere', 'cannot', "can't", "don't", "won't",
                  "wouldn't", "shouldn't", "couldn't", "isn't", "aren't",
                  "wasn't", "weren't", "haven't", "hasn't", "hadn't",
                  "didn't", "doesn't", "donot", "dont"}

INTENSIFIERS = {'very': 1.5, 'extremely': 2.0, 'really': 1.5, 'absolutely': 2.0,
                'completely': 1.5, 'totally': 1.5, 'highly': 1.5, 'strongly': 1.5,
                'utterly': 2.0, 'terribly': 1.5, 'so': 1.3, 'too': 1.3,
                'incredibly': 2.0, 'particularly': 1.3, 'exceptionally': 2.0}

NEGATIVE_WORDS = {
    # Severe (-3)
    'worst': -3, 'horrible': -3, 'terrible': -3, 'disgusting': -3,
    'unacceptable': -3, 'hopeless': -3, 'pathetic': -3, 'dreadful': -3,
    'outrageous': -3, 'inexcusable': -3, 'furious': -3,
    'harassment': -3, 'abuse': -3, 'robbery': -3, 'fraud': -3,
    'scam': -3, 'stolen': -3, 'cheated': -3,
    # Moderate (-2)
    'angry': -2, 'frustrated': -2, 'frustrating': -2, 'useless': -2,
    'ridiculous': -2, 'disappointed': -2, 'annoyed': -2, 'broken': -2,
    'damaged': -2, 'impossible': -2, 'unsafe': -2, 'dangerous': -2,
    'unhygienic': -2, 'malfunction': -2, 'stale': -2, 'smelly': -2,
    'fake': -2,
    # Mild (-1)
    'late': -1, 'delay': -1, 'problem': -1, 'issue': -1,
    'trouble': -1, 'difficult': -1, 'missing': -1, 'lost': -1,
    'urgent': -1, 'critical': -1, 'slow': -1, 'complaint': -1,
    'dirty': -1, 'leakage': -1, 'expired': -1,
}

POSITIVE_WORDS = {
    # Enthusiastic (+3)
    'excellent': 3, 'wonderful': 3, 'superb': 3, 'fantastic': 3,
    'amazing': 3, 'perfect': 3,
    # Moderate (+2)
    'thank': 2, 'appreciate': 2, 'grateful': 2, 'great': 2,
    'helpful': 2, 'efficient': 2, 'satisfied': 2, 'happy': 2,
    'pleased': 2, 'love': 2, 'best': 2, 'friendly': 2, 'polite': 2,
    'professional': 2, 'responsive': 2, 'supportive': 2,
    # Mild (+1)
    'good': 1, 'nice': 1, 'quick': 1, 'fast': 1, 'smooth': 1,
    'easy': 1, 'clean': 1, 'well': 1, 'better': 1, 'help': 1,
    'support': 1, 'resolve': 1, 'resolved': 1, 'solution': 1,
    'fixed': 1, 'comfortable': 1, 'timely': 1,
}

print(f'Lexicons loaded:')
print(f'  Negative words: {len(NEGATIVE_WORDS)}')
print(f'  Positive words: {len(POSITIVE_WORDS)}')
print(f'  Negation words: {len(NEGATION_WORDS)}')
print(f'  Intensifiers:   {len(INTENSIFIERS)}')

Lexicons loaded:
  Negative words: 50
  Positive words: 39
  Negation words: 27
  Intensifiers:   15


In [3]:
# ============================================================
# SENTIMENT ANALYSIS ENGINE
# ============================================================
def analyze_sentiment(text, unknown_words_log=None):
    """
    Token-by-token sentiment scanner with state tracking.
    
    State variables:
    - negate_next: If True, next sentiment word's score is flipped (× -1)
    - intensify_next: Multiplier for next sentiment word (1.0 = no boost)
    
    Parameters:
    - text: Input string
    - unknown_words_log: Optional list to collect unknown sentiment words
    """
    text_lower = text.lower()
    tokens = re.findall(r"[a-z]+'?[a-z]*", text_lower)
    
    score = 0.0
    word_count = 0
    negate_next = False
    intensify_next = 1.0
    negations_used = 0
    pos_words_found = []
    neg_words_found = []
    
    for token in tokens:
        multiplier = (-1.0 if negate_next else 1.0) * intensify_next
        
        if token in NEGATION_WORDS:
            negate_next = True
            negations_used += 1
            intensify_next = 1.0
            continue
        
        if token in INTENSIFIERS:
            intensify_next = INTENSIFIERS[token]
            continue
        
        if token in POSITIVE_WORDS:
            word_score = POSITIVE_WORDS[token] * multiplier
            score += word_score
            word_count += 1
            pos_words_found.append((token, word_score))
            negate_next = False
            intensify_next = 1.0
        elif token in NEGATIVE_WORDS:
            word_score = NEGATIVE_WORDS[token] * multiplier
            score += word_score
            word_count += 1
            neg_words_found.append((token, word_score))
            negate_next = False
            intensify_next = 1.0
        else:
            # ★ NEW WORD HANDLING ★
            # This token is not in any lexicon.
            # Current behavior: reset context, skip, no score impact.
            if unknown_words_log is not None:
                unknown_words_log.append(token)
            negate_next = False
            intensify_next = 1.0
    
    if word_count == 0:
        return {'label': 'Neutral', 'sub_label': 'Informational', 'score': 0.0,
                'neg_words': 0, 'pos_words': 0, 'negations': 0,
                'total_sentiment_words': 0}
    
    avg_score = score / word_count
    
    # Classify
    if avg_score < -0.5:
        sub_label = 'Angry / Frustrated'
    elif avg_score < -0.1:
        sub_label = 'Dissatisfied'
    elif avg_score > 0.5:
        sub_label = 'Appreciative'
    elif avg_score > 0.1:
        sub_label = 'Satisfied'
    else:
        sub_label = 'Informational'
    
    label = 'Negative' if avg_score < -0.1 else ('Positive' if avg_score > 0.1 else 'Neutral')
    
    return {
        'label': label,
        'sub_label': sub_label,
        'score': round(avg_score, 3),
        'neg_words': len(neg_words_found),
        'pos_words': len(pos_words_found),
        'negations': negations_used,
        'total_sentiment_words': word_count
    }

### Test sentiment analysis

In [4]:
test_texts = [
    "Thank you for your help, really appreciate it",
    "The problem is still not fixed, very disappointed",
    "WiFi is not working in the library",
    "Absolutely ridiculous and unacceptable behavior, extremely frustrating",
    "Please look into the water leakage in my room",
    "I hate this college, worst experience ever, useless administration",
    "Great work by the maintenance team, very helpful and quick response",
    "exam schedule not released yet",
    "someone stole my laptop from library",
]

print(f'{"Text":<55s} {"Label":<12s} {"Sub-label":<20s} {"Score":>8s}')
print('-' * 97)
for t in test_texts:
    r = analyze_sentiment(t)
    print(f'{t[:52]:<55s} {r["label"]:<12s} {r["sub_label"]:<20s} {r["score"]:>8.3f}')

Text                                                    Label        Sub-label               Score
-------------------------------------------------------------------------------------------------
Thank you for your help, really appreciate it           Positive     Appreciative            2.000
The problem is still not fixed, very disappointed       Negative     Angry / Frustrated     -1.667
WiFi is not working in the library                      Neutral      Informational           0.000
Absolutely ridiculous and unacceptable behavior, ext    Negative     Angry / Frustrated     -3.667
Please look into the water leakage in my room           Negative     Angry / Frustrated     -1.000
I hate this college, worst experience ever, useless     Negative     Angry / Frustrated     -2.500
Great work by the maintenance team, very helpful and    Positive     Appreciative            2.000
exam schedule not released yet                          Neutral      Informational           0.000
someone sto

---
## PART 7: NEW WORD HANDLING IN SENTIMENT ANALYSIS

### What happens when a new word appears?

When `analyze_sentiment()` encounters a token that isn't in any lexicon:
```
else:
    # Unknown word — not in NEGATION, INTENSIFIERS, POSITIVE, or NEGATIVE
    # Current behavior:
    negate_next = False    # Reset negation  ← PROBLEM!
    intensify_next = 1.0   # Reset intensifier ← PROBLEM!
    # No score impact       ← The word is invisible
```

### Why this is a problem:

In [5]:
# Problem demonstration
print('Example 1: Known word')
r = analyze_sentiment("The food is very bad")
print(f'  "bad" is in lexicon → score: {r["score"]}, label: {r["label"]}')
print()

print('Example 2: Unknown word (same meaning!)')
r = analyze_sentiment("The food is very atrocious")
print(f'  "atrocious" is NOT in lexicon → score: {r["score"]}, label: {r["label"]}')
print(f'  The strong negative word "atrocious" was completely ignored!')
print()

print('Example 3: Context loss with unknown words')
r = analyze_sentiment("I am not satisfied with the service")
print(f'  "not satisfied" → score: {r["score"]}, label: {r["label"]}')
print(f'  ("not" flips "satisfied" to negative = correct)')
print()

r = analyze_sentiment("I am not happy with the deplorable service")
print(f'  "not happy ... deplorable" → score: {r["score"]}, label: {r["label"]}')
print(f'  ("deplorable" is unknown, so half the negative sentiment is lost)')

Example 1: Known word
  "bad" is in lexicon → score: 0.0, label: Neutral

Example 2: Unknown word (same meaning!)
  "atrocious" is NOT in lexicon → score: 0.0, label: Neutral
  The strong negative word "atrocious" was completely ignored!

Example 3: Context loss with unknown words
  "not satisfied" → score: -2.0, label: Negative
  ("not" flips "satisfied" to negative = correct)

  "not happy ... deplorable" → score: -2.0, label: Negative
  ("deplorable" is unknown, so half the negative sentiment is lost)


### Solution 1: Log unknown words for review

Collect unknown words during analysis so an admin can add them later:

In [6]:
unknown_log = []
r = analyze_sentiment(
    "The deplorable and atrocious conditions are absolutely unacceptable",
    unknown_words_log=unknown_log
)

print(f'Result: {r["label"]} (score: {r["score"]})')
print(f'Unknown words found: {unknown_log}')
print()
print('These could be suggested as new lexicon entries:')
for w in unknown_log:
    print(f'  Add "{w}" → NEGATIVE_WORDS (suggest score: -2 or -3)')

Result: Negative (score: -6.0)
Unknown words found: ['the', 'deplorable', 'and', 'atrocious', 'conditions', 'are']

These could be suggested as new lexicon entries:
  Add "the" → NEGATIVE_WORDS (suggest score: -2 or -3)
  Add "deplorable" → NEGATIVE_WORDS (suggest score: -2 or -3)
  Add "and" → NEGATIVE_WORDS (suggest score: -2 or -3)
  Add "atrocious" → NEGATIVE_WORDS (suggest score: -2 or -3)
  Add "conditions" → NEGATIVE_WORDS (suggest score: -2 or -3)
  Add "are" → NEGATIVE_WORDS (suggest score: -2 or -3)


### Solution 2: Fuzzy / Synonym matching

If a word is unknown, check if it contains a known word as a substring:

In [7]:
def analyze_sentiment_fuzzy(text):
    """
    Enhanced version: if a word isn't in the lexicon,
    check if any known word is a substring of it.
    Example: "atrocious" → contains "atro"? No. 
    But "disrespectful" → contains "respect"? Not helpful.
    
    Better: build a set of known word STEMS for matching.
    "disrespectful" → stem → "disrespect" (contains "respect"? no)
    
    A more practical approach: maintain a SYNONYM MAP
    "deplorable" → synonym of "terrible" → score -3
    "atrocious"  → synonym of "horrible" → score -3
    "appalling"  → synonym of "shocking" → score -2
    "dreadful"   → synonym of "terrible" → score -3  (already in our list!) 
    
    But building a full synonym map requires WordNet or manual entry.
    For now: logging is the simplest practical solution.
    """
    pass

---
## PART 8: COMPLETE SENTIMENT-TO-PRIORITY PIPELINE

When a complaint is submitted, the system runs it through all components:

In [8]:
import sys
sys.path.insert(0, r'E:/Project-VI/workspace/ComplaintMgmtSystem/ml')
from classifier import MultinomialNB
model = MultinomialNB.load(r'E:/Project-VI/workspace/ComplaintMgmtSystem/data/model_params.json')
print('Loaded real trained model: %d classes, %d vocab' % (len(model.classes), model.vocab_size))

# Category decoder: numeric ID to human name
CAT_DECODER = {0: 'IT Support', 1: 'Hostels', 2: 'Academics', 3: 'Fees / Finance', 4: 'Maintenance', 5: 'Transport',
               6: 'Security / Discipline', 7: 'Administration', 8: 'Library', 9: 'Canteen'}

Loaded real trained model: 10 classes, 2324 vocab


In [9]:
def sentiment_priority_boost(sentiment_label, current_priority):
    """
    If sentiment is Negative, escalate priority one level.
    This ensures frustrated users get faster attention.
    """
    boosts = {
        'Negative': {'Low': 'Medium', 'Medium': 'High', 'High': 'High'},
        'Neutral':  {'Low': 'Low',   'Medium': 'Medium', 'High': 'High'},
        'Positive': {'Low': 'Low',   'Medium': 'Medium', 'High': 'High'},
    }
    return boosts.get(sentiment_label, {}).get(current_priority, current_priority)


def full_pipeline(complaint_text, base_priority='Medium'):
    """Simulate the full NLP pipeline when a complaint is submitted."""
    print(f'Complaint: "{complaint_text}"')
    print(f'Base priority: {base_priority}')
    print()
    
    # 1. Categorization
    # (normally uses trained model; here we simulate)
    pred, probs = model.predict_with_proba(complaint_text)
    category = CAT_DECODER[pred]
    confidence = probs[pred]
    print(f'1. Category: {category} (confidence: {confidence:.1%})')
    
    # 2. Sentiment analysis
    sentiment = analyze_sentiment(complaint_text)
    print(f'2. Sentiment: {sentiment["label"]} ({sentiment["sub_label"]}), score: {sentiment["score"]}')
    print(f'   Negations used: {sentiment["negations"]}')
    
    # 3. Priority boost
    boosted = sentiment_priority_boost(sentiment['label'], base_priority)
    print(f'3. Priority: {base_priority} → {boosted}')
    print()
    return {
        'category': category,
        'confidence': confidence,
        'sentiment': sentiment,
        'final_priority': boosted
    }

full_pipeline("The wifi is very slow and not working properly in the library", 'Medium')
print()
full_pipeline("Absolutely disgusting food, extremely disappointed, worst experience ever", 'Low')

Complaint: "The wifi is very slow and not working properly in the library"
Base priority: Medium

1. Category: IT Support (confidence: 99.3%)
2. Sentiment: Negative (Angry / Frustrated), score: -1.5
   Negations used: 1
3. Priority: Medium → High


Complaint: "Absolutely disgusting food, extremely disappointed, worst experience ever"
Base priority: Low

1. Category: Canteen (confidence: 97.8%)
2. Sentiment: Negative (Angry / Frustrated), score: -4.333
   Negations used: 0
3. Priority: Low → Medium



{'category': 'Canteen',
 'confidence': 0.9777077705835364,
 'sentiment': {'label': 'Negative',
  'sub_label': 'Angry / Frustrated',
  'score': -4.333,
  'neg_words': 3,
  'pos_words': 0,
  'negations': 0,
  'total_sentiment_words': 3},
 'final_priority': 'Medium'}